# Change of Basis for Periodic Splines
The data samples at the integers are represented with circles and stem lines. The sample at the origin, as well as its periodized replicates, is indicated by a red circle and stem line. The thin colored curves are the weighted and shifted basis components whose sum is the curve depicted in thick blue.

In [ ]:
# Load the required libraries
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

import splinekit as sk # This library

# Setup
max_degree = 9 # Maximal spline degree
max_samples = 15 # Maximal support

# Initial random periodic cubic spline
s0 = sk.PeriodicSpline1D.from_spline_coeff(np.random.standard_normal(6), degree = 3)

# Plot
def update_plot (
    degree = 3,
    samples = 6,
    basis = 0
):
    global s0

    # Update of the spline
    f = s0.get_samples(0, support_length = s0.period)
    if s0.period < samples:
        f = np.append(f, np.random.standard_normal(samples - len(f)))
    else:
        f = f[ : samples]
    s0 = sk.PeriodicSpline1D.from_samples(f, degree = degree)

    # Decomposition according to the basis
    weights = np.copy(s0.spline_coeff)
    if 0 == basis: # BASIC
        sk.change_basis_p(
            weights,
            degree = degree,
            source_basis = sk.Bases.BASIC,
            target_basis = sk.Bases.BASIC
        )
        bases = [
            sk.PeriodicSpline1D.periodized_b_spline(
                period = s0.period,
                degree = degree
            ).delayed_by(k).times(wk)
            for (k, wk) in enumerate(weights)
        ]
    elif 1 == basis: # CARDINAL
        sk.change_basis_p(
            weights,
            degree = degree,
            source_basis = sk.Bases.BASIC,
            target_basis = sk.Bases.CARDINAL
        )
        bases = [
            sk.PeriodicSpline1D.periodized_cardinal_b_spline(
                period = s0.period,
                degree = degree
            ).delayed_by(k).times(wk)
            for (k, wk) in enumerate(weights)
        ]
    elif 2 == basis: # DUAL
        sk.change_basis_p(
            weights,
            degree = degree,
            source_basis = sk.Bases.BASIC,
            target_basis = sk.Bases.DUAL
        )
        bases = [
            sk.PeriodicSpline1D.periodized_dual_b_spline(
                period = s0.period,
                dual_degree = degree,
                primal_degree = degree
            ).delayed_by(k).times(wk)
            for (k, wk) in enumerate(weights)
        ]
    elif 3 == basis: # ORTHONORMAL
        sk.change_basis_p(
            weights,
            degree = degree,
            source_basis = sk.Bases.BASIC,
            target_basis = sk.Bases.ORTHONORMAL
        )
        bases = [
            sk.PeriodicSpline1D.periodized_orthonormal_b_spline(
                period = s0.period,
                degree = degree
            ).delayed_by(k).times(wk)
            for (k, wk) in enumerate(weights)
        ]

    # Dynamic range
    image = {b.image() for b in bases}
    image.add(s0.image())
    plotrange = sk.interval.Interval.enclosure(image)
    plotrange = sk.interval.Closed((
        plotrange.midpoint - 0.55 * plotrange.diameter,
        plotrange.midpoint + 0.55 * plotrange.diameter
    ))

    # Plot of the spline
    subplot = plt.subplots()
    s0.plot(
        subplot,
        plotrange = plotrange,
        plotpoints = 200 + 1,
        curve_fmt = "-C0",
        knot_marker = ""
    )

    # Independent plot of each weighted basis component
    for (k, b) in enumerate(bases):
        b.plot(
            subplot,
            plotpoints = 200 + 1,
            curve_fmt = "-C" + str(1 + k % 9),
            curve_lw = 0.25,
            curve_markerfmt = "",
            curvestem_linefmt = "None",
            knot_marker = "",
            periodbound_markerfmt = "",
            periodboundstem_linefmt = "None"
        )
    plt.show()

widgets.interactive(
    update_plot,
    degree = (0, max_degree),
    samples = (1, max_samples),
    basis = widgets.RadioButtons(
        options = [
            ("BASIC", 0),
            ("CARDINAL", 1),
            ("DUAL", 2),
            ("ORTHONORMAL", 3)
        ],
        value = 0,
        description = "Basis:",
        disabled = False
    )
)
